# Adding Metadata
1. who sen't the message? - sender_priority
2. what time? - hour_of_day, day_of_week
3. is_group_chat - more likely to spam
4. how often/how much does the sender text? - average_messages_per_day
5. recent activity - recent_activity

In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
df = pd.read_csv('chats_sample_features.csv')
df = df[df['label'] != -1]

# Get unique senders (assuming the sender column is named 'sender')
# If the column name is different, please let me know.
unique_senders = df['sender'].unique()

print("Unique Senders:")
for sender in unique_senders:
    print(sender)

display(df.head())

sender_importance

In [ ]:
sender_importance = {
      # Person_N (from earlier scoring)
      "Person_1":  2,  "Person_2":  4,  "Person_3":  2,  "Person_4":  1,
      "Person_5":  5,  "Person_6":  5,  "Person_7":  5,  "Person_8":  5,
      "Person_9":  5,  "Person_10": 5,  "Person_11": 1,  "Person_12": 1,
      "Person_13": 3,  "Person_14": 2,
      # Groups (by chat type)
      "group_1":   5,  # family emergency group
      "group_2":   4,  # work / team channel
      "group_3":   3,  # close friends
      "group_4":   2,  # society / neighbours
      "group_5":   1,  # social / memes
  }

# Add the 'sender_importance' column to the DataFrame
df['sender_importance'] = df['sender'].map(sender_importance)

# Display the first few rows with the new column to verify
display(df.head())

time_of_day and day_of_week

In [ ]:
# Convert 'timestamp' to datetime objects, coercing errors to NaT
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Identify rows where 'timestamp' is NaT
nan_mask = df['timestamp'].isna()

# For these rows, combine 'date' and 'time' to create a new datetime string
# Using .astype(str) to handle potential mixed types in 'date' or 'time'
combined_dt_str = df['date'].astype(str) + ' ' + df['time'].astype(str)

# Attempt to parse the combined 'date' and 'time' strings into datetime objects
# infer_datetime_format=True helps with varied date/time formats
parsed_from_dt_cols = pd.to_datetime(combined_dt_str, errors='coerce', infer_datetime_format=True)

# Fill the NaT values in 'timestamp' with the newly parsed datetimes
df['timestamp'] = df['timestamp'].fillna(parsed_from_dt_cols)

# Extract 'hour_of_day' from the (now cleaner) 'timestamp' column
df['hour_of_day'] = df['timestamp'].dt.hour

# Extract 'day_of_week' (Monday=0, Sunday=6) from the 'timestamp' column
df['day_of_week'] = df['timestamp'].dt.dayofweek

# Convert day numbers to full day names
day_name_map = {
    0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
    4: 'Friday', 5: 'Saturday', 6: 'Sunday'
}
df['day_of_week'] = df['day_of_week'].map(day_name_map)

# Display the first few rows with the new columns to verify
display(df.head())

is_group_chat

In [ ]:
# Create 'is_group_chat' column: 1 if sender starts with 'group_', 0 otherwise
df['is_group_chat'] = df['sender'].str.startswith('group_').astype(int)

# Display the first few rows with the new column to verify
display(df.head())

average_messages_per_day

In [ ]:
# Calculate the number of messages sent by each sender on each day
messages_per_sender_per_day = df.groupby(['sender', df['timestamp'].dt.date]).size().reset_index(name='daily_message_count')

# Calculate the average number of messages per day for each sender
average_messages_per_sender = messages_per_sender_per_day.groupby('sender')['daily_message_count'].mean().reset_index(name='average_messages_per_day')

# Merge this average back into the original DataFrame based on the 'sender'
df = df.merge(average_messages_per_sender, on='sender', how='left')

# Display the first few rows with the new column to verify
display(df.head())

recent_message_frequency

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
df['recent_message_frequency'] = (
    df.groupby('sender')
      .rolling('10min', on='timestamp')
      .count()['message']
      .reset_index(drop=True)
)
df['recent_message_frequency'].value_counts()

In [ ]:
df.describe()

In [ ]:
df.head()

# Feature Exploration

In [ ]:
df.groupby('label')['sender_importance'].mean()

In [ ]:
pd.crosstab(df['is_group_chat'], df['label'])

In [ ]:
df.groupby('label')['average_messages_per_day'].mean()

In [ ]:
df.groupby('label')['hour_of_day'].mean()

In [ ]:
import seaborn as sns
sns.histplot(
    data=df,
    x='hour_of_day',
    hue='label',
    bins=24,
    multiple='stack'
)

# Vectorization

In [ ]:
#vectorize the message contents
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=3000
)

X_text = vectorizer.fit_transform(df['message'])

In [ ]:
metadata_features = [
    'sender_importance',
    'is_group_chat',
    'hour_of_day',
    'average_messages_per_day'
]

X_meta = df[metadata_features]

In [ ]:
from scipy.sparse import csr_matrix
X_meta_sparse = csr_matrix(X_meta.values)

In [ ]:
from scipy.sparse import hstack
X_combined = hstack([X_text, X_meta_sparse])

In [ ]:
print(X_combined)

## Note on Vectorization and X_combined

After vectorizing the message contents into `X_text` using `TfidfVectorizer`, we extracted additional metadata features such as `sender_importance`, `is_group_chat`, `hour_of_day`, and `average_messages_per_day` into a new DataFrame `X_meta`. This `X_meta` DataFrame was then converted into a sparse matrix, `X_meta_sparse`. Finally, `X_combined` was created by horizontally stacking the `X_text` (text features) and `X_meta_sparse` (metadata features).

The output of `X_combined` represents a `Compressed Sparse Row (CSR)` matrix. It has a shape of (454, 736), meaning it contains 454 samples (rows) and 736 features (columns). The output also shows that it has 2547 stored elements, indicating that most of its values are zero, which is typical for sparse matrices. The `Coords` and `Values` sections display a few examples of where non-zero elements are located (row, column coordinates) and their corresponding values.

# Train-Test Split

In [ ]:
y = df['label']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# stratify=y preserves class proportions.

# Logistic Regression with Metadata

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

# Model Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
lg_precision = 0.83
lg_recall = 0.62
lg_f1 = 0.71

Metric Comparisions

Values taken from Bypass_DND_Baseline notebook

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

comparison_df = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'F1-Score'],
    'Baseline': [0.40, 0.57, 0.47],
    'Hybrid': [0.83, 0.62, 0.71]
})

comparison_df.set_index('Metric').plot(kind='bar', figsize=(8,5))

plt.title('Baseline vs Hybrid Model Performance (Urgent Class)')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0, 1)

plt.show()

Contextual features dramatically improved precision
and overall balance.

In [ ]:
error_df = pd.DataFrame({
    'Error Type': ['False Positives', 'False Negatives'],
    'Baseline': [6, 3],
    'Hybrid': [1, 3]
})

error_df.set_index('Error Type').plot(kind='bar', figsize=(8,5))

plt.title('Error Comparison: Baseline vs Hybrid')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.show()

The contextual model greatly reduced unnecessary
DND interruptions without increasing missed urgent messages.

**Error Analysis**

In [ ]:
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X_combined,
    y,
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
results = df.loc[test_idx].copy()

results['actual'] = y_test.values
results['predicted'] = y_pred

In [ ]:
false_positives = results[
    (results['actual'] == 0) &
    (results['predicted'] == 1)
]

In [ ]:
pd.set_option('display.max_colwidth', None)

false_positives[['message', 'sender_importance',
                 'is_group_chat', 'hour_of_day']]

**Observations:**


1.   The hybrid model appeared to weigh sender importance
strongly when interpreting emotionally concerned messages,
increasing the likelihood of urgency prediction.
2.   Longer conversational messages may produce richer textual
signals, allowing the model to form stronger confidence
around urgency-related interpretations.
3. Urgency is subjective - this is a unique corner case where the concerned language+sender importance may result in a message that a user might genuinely want to bypass DND



In [ ]:
false_negatives = results[
    (results['actual'] == 1) &
    (results['predicted'] == 0)
]
pd.set_option('display.max_colwidth', None)
false_negatives[['message', 'sender_importance',
                 'is_group_chat', 'hour_of_day']]

# False Negative Observations

## Observation 1 — Vague Emotional Language Is Difficult to Classify

Messages like:

```text id="h9zvv9"
"I'm not okay right now"
```

express emotional distress but lack explicit urgency indicators such as:

* emergency-related keywords
* direct requests for action
* high-priority intent markers

This makes urgency detection challenging for text-based models.

---

## Observation 2 — Contextual Features May Override Strong Textual Signals

Messages such as:

```text id="aqb9p4"
"This can't wait, call me now"
```

contain strong urgency wording, yet were still classified as non-urgent.

This suggests the hybrid model may place significant weight on contextual metadata such as:

* sender importance
* historical communication patterns

potentially suppressing urgent predictions from low-priority senders.

---

## Observation 3 — Sender Importance Appears Highly Influential

Multiple false negatives involved relatively low sender importance scores.

This indicates that the model may strongly associate urgency with trusted or high-priority contacts, reducing sensitivity toward urgent messages originating from less important senders.

---

## Observation 4 — Temporal Signals Alone Were Insufficient

Some false negatives occurred during unusual hours (e.g., 3–4 AM), which might intuitively imply higher urgency.

However, the model still classified them as non-urgent, suggesting that:

* time-of-day features had weaker influence
  compared to
* sender-based contextual signals.

---

## Observation 5 — Administrative or Structured Content Can Dilute Urgency

Long structured messages containing:

* phone numbers
* labels
* contact details
* repetitive informational text

may dilute the representation of urgent phrases within the TF-IDF feature space.

For example, an urgent opening phrase like:

```text id="7a3vny"
"I need help right now"
```

can become statistically overshadowed by large amounts of low-signal administrative content.

---

## Observation 6 — Urgency Detection Is Inherently Subjective

Several false negatives highlight the ambiguity of urgency itself.

Some messages may:

* appear emotionally urgent
* contain alarming wording
  yet still not represent situations a user would want to interrupt Do Not Disturb mode for.

This reflects a broader challenge in designing intelligent notification systems:

* urgency is highly dependent on personal context and user preference.

---

## Observation 7 — Hybrid Models Introduce Tradeoffs

The hybrid model significantly reduced false positives, improving notification reliability.

However, this improvement may come at the cost of occasionally underpredicting urgency when contextual metadata conflicts with textual urgency cues.

This demonstrates the tradeoff between:

* minimizing unnecessary interruptions
  and
* maximizing urgent message detection.


# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

In [ ]:
rf_preds = rf_model.predict(X_test)

**Evaluation**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score

print("Random Forest Classification Report:")
print(classification_report(y_test, rf_preds))

print("\nRandom Forest Confusion Matrix:")
print(confusion_matrix(y_test, rf_preds))

# Calculate individual metrics
rf_precision = precision_score(y_test, rf_preds)
rf_recall = recall_score(y_test, rf_preds)
rf_f1 = f1_score(y_test, rf_preds)

print(f"\nRandom Forest Precision: {rf_precision:.2f}")
print(f"Random Forest Recall: {rf_recall:.2f}")
print(f"Random Forest F1-Score: {rf_f1:.2f}")

In [ ]:
lg_accuracy = 0.96
lg_fp = 1
lg_fn = 3

rf_accuracy = 0.97
rf_fp = 1
rf_fn = 2

comparison_rf_lg_df_extended = pd.DataFrame({
    'Metric': ['Precision', 'Recall', 'F1-Score', 'Accuracy', 'False Positives', 'False Negatives'],
    'Logistic Regression': [lg_precision, lg_recall, lg_f1, lg_accuracy, lg_fp, lg_fn],
    'Random Forest': [rf_precision, rf_recall, rf_f1, rf_accuracy, rf_fp, rf_fn]
})

display(comparison_rf_lg_df_extended)

In [ ]:
metrics_to_plot = ['Precision', 'Recall', 'F1-Score', 'Accuracy']
comparison_rf_lg_df_extended[comparison_rf_lg_df_extended['Metric'].isin(metrics_to_plot)].set_index('Metric').plot(kind='bar', figsize=(12, 7))
plt.title('Logistic Regression vs Random Forest Performance (Extended Metrics)')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.show()

errors_to_plot = ['False Positives', 'False Negatives']
comparison_rf_lg_df_extended[comparison_rf_lg_df_extended['Metric'].isin(errors_to_plot)].set_index('Metric').plot(kind='bar', figsize=(8, 5), colormap='viridis')
plt.title('Logistic Regression vs Random Forest Error Comparison')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(loc='upper right')
plt.show()

In [ ]:
comparison_rf_lg_df_extended.set_index('Metric').plot(kind='bar', figsize=(10, 6))
plt.title('Logistic Regression vs Random Forest Performance (Urgent Class)')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.show()

Recall improved without increasing false positives. That suggests that the Random Forest captured contextual interactions more effectively without sacrificing reliability.